In [1]:
import json
import os
from pathlib import Path

import pandas as pd

In [2]:
RESULTS_DIR = Path(os.path.expanduser("~/scFM_eval/results"))
TABLE_DIR = Path("./tables")
TABLE_DIR.mkdir(parents=True, exist_ok=True)

CLASSIFIER = "randomforest"
EXCLUDE_EXPS = {"luad1", "luad_cancer_stage"}
METRIC_COLS = ["AUC", "AUPRC", "Accuracy", "F1", "Precision", "Recall"]
TABLE_COLUMNS = ["model", "group", *METRIC_COLS, "strategy", "exp", "task"]


def load_classification_metrics(results_dir: Path) -> pd.DataFrame:
    """Load fold-level classification metrics from CSV or JSON exports."""
    csv_path = results_dir / "classification.metrics.csv"
    json_path = results_dir / "classification.metrics.json"
    if csv_path.exists():
        return pd.read_csv(csv_path)
    if json_path.exists():
        payload = json.loads(json_path.read_text())
        return pd.DataFrame(payload["records"])
    raise FileNotFoundError(
        f"No classification.metrics.csv or .json in {results_dir}"
    )


raw = load_classification_metrics(RESULTS_DIR)
metrics = raw[raw["classifier"] == CLASSIFIER].copy()
metrics = metrics[~metrics["exp"].isin(EXCLUDE_EXPS)].copy()
metrics = metrics.dropna(subset=["model_display", "exp_display", "group"])
metrics = metrics[
    ~metrics["model_display"].str.contains("finetune", case=False, na=False)
]

print(
    f"{metrics['model_display'].nunique()} models, "
    f"{metrics['exp_display'].nunique()} tasks, "
    f"{len(metrics)} fold rows from {RESULTS_DIR}"
)
print("strategies:", sorted(metrics["strategy"].unique()))

17 models, 7 tasks, 1734 fold rows from /home/haitham/scFM_eval/results
strategies: ['MIL', 'avg', 'vote']


In [3]:
all_df = (
    metrics.groupby(
        ["model_display", "group", "strategy", "exp", "exp_display"],
        as_index=False,
    )[METRIC_COLS]
    .mean()
    .rename(columns={"model_display": "model", "exp_display": "task"})
)
all_df = all_df.round(3)
all_df = all_df[TABLE_COLUMNS].sort_values(
    ["strategy", "exp", "group", "model"]
).reset_index(drop=True)
all_df

,model,group,AUC,AUPRC,Accuracy,F1,Precision,Recall,strategy,exp,task
0,HVG,Baseline,0.633,0.587,0.821,0.564,0.567,0.583,MIL,brca_chemo,Treatment Naive vs Neoadjuvant Chemo
1,PCA [100],Baseline,0.833,0.773,0.821,0.516,0.509,0.550,MIL,brca_chemo,Treatment Naive vs Neoadjuvant Chemo
2,PCA [20],Baseline,0.833,0.729,0.871,0.744,0.790,0.733,MIL,brca_chemo,Treatment Naive vs Neoadjuvant Chemo
3,PCA [50],Baseline,0.767,0.633,0.821,0.581,0.616,0.583,MIL,brca_chemo,Treatment Naive vs Neoadjuvant Chemo
4,scVI,Baseline,0.833,0.673,0.771,0.498,0.502,0.517,MIL,brca_chemo,Treatment Naive vs Neoadjuvant Chemo
...,...,...,...,...,...,...,...,...,...,...,...
352,STATE,Other,0.900,0.950,0.750,0.707,0.750,0.750,vote,melanoma_response,IO Response
353,scConcept,Other,0.900,0.950,0.750,0.707,0.750,0.750,vote,melanoma_response,IO Response
354,scFoundation,Other,0.850,0.917,0.750,0.707,0.750,0.750,vote,melanoma_response,IO Response
355,scGPT,scGPT,0.900,0.950,0.850,0.840,0.900,0.850,vote,melanoma_response,IO Response


In [4]:
all_df.to_csv(TABLE_DIR / "Table8_classification_metrics.csv", index=False)
print(f"Wrote {len(all_df)} rows to {TABLE_DIR / 'Table8_classification_metrics.csv'}")

Wrote 357 rows to tables/Table8_classification_metrics.csv


In [5]:
all_df.strategy.value_counts()

strategy
MIL     119
avg     119
vote    119
Name: count, dtype: int64